# Frequent Value (Mode) Imputation

For a **categorical** column, there's no mean/median to fall back on — the equivalent is filling missing values with the **most frequent category (the mode)**. This notebook checks that assumption too: is it actually safe to assume a missing `GarageQual`/`FireplaceQu` behaves like the most common category?

## 1. Setup and Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('train.csv',usecols=['GarageQual','FireplaceQu','SalePrice'])

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
df.head()

In [ ]:
df.isnull().mean()*100

## 2. `GarageQual`

In [ ]:
df['GarageQual'].value_counts().plot(kind='bar')

`.mode()` confirms the single most common category — `'TA'` (typical/average) here.

In [ ]:
df['GarageQual'].mode()

### Checking the Key Assumption

Mode imputation implicitly assumes: *rows missing this value would have looked like the most common category, had it been recorded.* Comparing `SalePrice`'s distribution for houses that actually have `'TA'` against houses where `GarageQual` is missing tests that assumption directly — if the two curves look similar, treating missing as `'TA'` is reasonable; if they diverge, missing garages might mean something different (e.g. no garage at all).

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

df[df['GarageQual']=='TA']['SalePrice'].plot(kind='kde', ax=ax)

df[df['GarageQual'].isnull()]['SalePrice'].plot(kind='kde', ax=ax, color='red')

lines, labels = ax.get_legend_handles_labels()
labels = ['Houses with TA', 'Houses with NA']
ax.legend(lines, labels, loc='best')

plt.title('GarageQual')

In [ ]:
temp = df[df['GarageQual']=='TA']['SalePrice']

In [ ]:
df['GarageQual'].fillna('TA', inplace=True)

Re-checking the category counts after filling — `'TA'`'s bar should have grown by exactly the number of previously-missing rows.

In [ ]:
df['GarageQual'].value_counts().plot(kind='bar')

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)


temp.plot(kind='kde', ax=ax)

# distribution of the variable after imputation
df[df['GarageQual'] == 'TA']['SalePrice'].plot(kind='kde', ax=ax, color='red')

lines, labels = ax.get_legend_handles_labels()
labels = ['Original variable', 'Imputed variable']
ax.legend(lines, labels, loc='best')

# add title
plt.title('GarageQual')

## 3. `FireplaceQu` — Same Process

In [ ]:
df['FireplaceQu'].value_counts().plot(kind='bar')

In [ ]:
df['FireplaceQu'].mode()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

df[df['FireplaceQu']=='Gd']['SalePrice'].plot(kind='kde', ax=ax)

df[df['FireplaceQu'].isnull()]['SalePrice'].plot(kind='kde', ax=ax, color='red')

lines, labels = ax.get_legend_handles_labels()
labels = ['Houses with Gd', 'Houses with NA']
ax.legend(lines, labels, loc='best')

plt.title('FireplaceQu')

In [ ]:
temp = df[df['FireplaceQu']=='Gd']['SalePrice']

In [ ]:
df['FireplaceQu'].fillna('Gd', inplace=True)

In [ ]:
df['FireplaceQu'].value_counts().plot(kind='bar')

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)


temp.plot(kind='kde', ax=ax)

# distribution of the variable after imputation
df[df['FireplaceQu'] == 'Gd']['SalePrice'].plot(kind='kde', ax=ax, color='red')

lines, labels = ax.get_legend_handles_labels()
labels = ['Original variable', 'Imputed variable']
ax.legend(lines, labels, loc='best')

# add title
plt.title('FireplaceQu')

Worth a specific gut-check here: `FireplaceQu` being missing plausibly just means **"no fireplace"** — conceptually quite different from "average fireplace quality." If the two `SalePrice` curves in the plot above diverge noticeably, that's a sign mode imputation may not be the right call for this particular column, even though it's a completely mechanical fix to apply.

## 4. Doing the Same Thing With `SimpleImputer`

`strategy='most_frequent'` computes the mode **per column independently** — one imputer here handles both `GarageQual` and `FireplaceQu` at once, each getting its own learned mode.

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['SalePrice']),df['SalePrice'],test_size=0.2)

In [ ]:
from sklearn.impute import SimpleImputer

In [ ]:
imputer = SimpleImputer(strategy='most_frequent')

### ⚠️ Bug: Transforming `X_train` Again Instead of `X_test`

`X_test = imputer.transform(X_train)` re-transforms **`X_train`** and assigns the result to `X_test` — `X_test` never actually gets imputed at all. This is exactly the "fit on train, transform on train *and* test" pattern from earlier notebooks — just a variable-name slip here rather than a wrong method call.

In [ ]:
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)   # fixed: transform X_test, not X_train again


In [ ]:
imputer.statistics_

## Summary

| Task | Method |
|---|---|
| Find the mode (manual) | `df['col'].mode()` |
| Fill with the mode (manual) | `df['col'].fillna(mode_value, inplace=True)` |
| Fill with the mode (sklearn, reusable) | `SimpleImputer(strategy='most_frequent')` |
| Check the imputation assumption | compare the target's distribution: "has the mode category" vs. "was missing" |
| See what each column's learned mode was | `imputer.statistics_` |

### Watch out for
Always double-check `X_test` is transformed with the **fitted** imputer, not accidentally re-run on `X_train` — an easy copy-paste slip once train/test variable names start repeating across cells, exactly as happened here.